# 逐档增益旁路/抵消

先静默选档，再录音，最后一批完成。只开这一本音频Notebook。箱盖、麦克风、两只音响及Codec音量保持原标定条件。手机只在录音时外放。最多8档，直放1项加每档2项；录2秒时最多发声40.8秒，整轮预留10–15分钟（分析时间依板卡而定）。持续尖叫立即断音响电源；输入框不是实时急停。

## 第1格：静默选档
音响保持断电，运行下面一格一次。列出可测档位且提示选档完成才继续。不需填写参数，不需启动手机。失败则返回打印的summary.json，不进入第2格。

In [ ]:
from pathlib import Path
import sys, os, importlib
BOARD = Path('/home/xilinx/jupyter_notebooks/base/audio/audio_replay')
os.chdir(str(BOARD))
if str(BOARD) not in sys.path: sys.path.insert(0, str(BOARD))
import replay_tools, replay_session, sweep_plan, sweep_session
for module in (replay_tools, replay_session, sweep_plan, sweep_session): importlib.reload(module)
if 'replay_trial' in sys.modules: importlib.reload(sys.modules['replay_trial'])
plan_path = sweep_session.prepare_sweep()

## 第2格：录音2秒
确认第1格成功；音响断电。运行本格后，等RECORD提示出现，手机在原位置播放语音，输入RECORD回车立即录2秒。录完停止手机外放。已有source.wav自动复用。

In [ ]:
assert plan_path is not None, '选档失败，停止并返回第1格摘要'
source_path = replay_session.record_source()

## 第3格：逐档听音
停止手机外放，恢复音响供电。执行本格后，现场就绪输入READY。每项配置完成才提示START；输入后立即播放并采样，约2.4秒后静音。先直放，再每档抵消→旁路，全部档位事先列出。

每项结束输入：0无声、1清楚语音且结束安静、2杂音失真、3啸叫增长、4结束仍响。仅1继续；其他输入或数字异常停止整批，保留已完成配对，不再升档。若旁路出现啸叫，此档抵消项已先完成。

结束只下载最后打印路径的summary.json，WAV/NPZ留板。不要Run All。

In [ ]:
result = sweep_session.run_sweep(plan_path)